# Data loadings and preparations

> Fill in a module description here


In [ ]:
#| default_exp data.__init__

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import importlib
from dataclasses import dataclass
from typing import Dict
from omegaconf import OmegaConf, DictConfig
from dataclasses import is_dataclass, asdict

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.distributed as dist

from fastcore.utils import *
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import *
from flwr_datasets.preprocessor import Merger

from fedai.data.vision import VisionBlock
from fedai.core import get_clean_kwargs

In [ ]:
#| export
def get_preprocessor(cfg):
    dataset_dict = {"train": "train", "test": "test"}
    if cfg.data.name == "tinyimagenet":
        dataset_dict = {"train": "train", "valid": "valid"}
        
    merger = Merger(
        merge_config={
             "train": tuple(list(key for key in dataset_dict.keys())),
            }
        )
    return merger

In [ ]:
#| export
def init_data(cfg):
    if cfg.data.name != "mnist_rotated_batched":
        module = importlib.import_module("flwr_datasets.partitioner")
    else: 
        module = importlib.import_module("fedai.data.vision.partitioners")
    
    partitioner_cls = getattr(module, cfg.partitioner.cls)
    kwargs = get_clean_kwargs(cfg.partitioner)
    kwargs.pop("cls", None)
    
    partitioner = partitioner_cls(**kwargs, seed= cfg.random_seed)    
    preprocessor = get_preprocessor(cfg)
    
    fds = FederatedDataset(dataset= cfg.data.hf_name, 
                           partitioners= {"train": partitioner},
                           preprocessor= preprocessor,
                           seed= cfg.random_seed)
    
    return fds

In [ ]:
#| hide
from fedai.cfgs.data import MNISTRotatedPatchedConfig, CIFAR10Config
from fedai.cfgs.partitioners import RotatedBatchedConfig, PathologicalConfig
from omegaconf import OmegaConf
import numpy as np


In [ ]:
#| hide
cfg = OmegaConf.create({})
cfg.data = CIFAR10Config()
cfg.partitioner = PathologicalConfig()
cfg.random_seed = 42    
fds = init_data(cfg)


In [ ]:
#| hide
partition = fds.load_partition(partition_id= 16, split= "train")
partition

Dataset({
    features: ['img', 'label'],
    num_rows: 3000
})

In [ ]:
#| hide
np.unique(partition["label"])

array([6, 7])

In [ ]:
#| export
def get_block(cfg, client_id, fds= None, train=True):

    if cfg.data.modality.lower() == "vision":
        return VisionBlock(cfg.data, client_id, train= train, fds= fds, seed= cfg.random_seed)

In [ ]:
#| hide
block = get_block(cfg, client_id= 1, fds= fds, train=True)

In [ ]:
#| hide
type(block[0]['image'])

torch.Tensor

In [ ]:
#| hide
import matplotlib.pyplot as plt
import numpy as np

def show_normalized_img(img_tensor, mean, std):
    # img_tensor shape: (C, H, W)
    img = img_tensor.clone().numpy().transpose(1, 2, 0)
    
    # Reverse the normalization: (x * std) + mean
    img = img * np.array(std) + np.array(mean)
    
    # Clip to [0, 1] just in case of rounding errors
    img = np.clip(img, 0, 1)
    
    plt.imshow(img)
    plt.show()

# Usage with your block:
# show_normalized_img(block[0]['image'], cfg.data.mean, cfg.data.std)
idx = np.random.randint(0, len(block))
block[idx]['image'].mean(dim=(1, 2)), block[idx]['image'].std(dim=(1, 2))

(tensor([-0.1281, -0.2441, -0.2947]), tensor([0.5976, 0.8873, 1.3872]))

In [ ]:
#| hide
imgs = torch.stack([block[i]['image'] for i in range(len(block))], dim=0)  # (N, C, H, W)

print(imgs.mean(dim=[0, 2, 3]))  # should be close to [0, 0, 0]
print(imgs.std(dim=[0, 2, 3], unbiased=False))   # should be close to [1, 1, 1]

tensor([-0.0592, -0.0386, -0.0575])
tensor([1.2339, 1.2350, 1.2956])


In [ ]:
#| hide
image, label = block[2]["image"], block[2]["label"]
print(image.shape, image.min(), image.max())
plt.imshow(image.permute(1, 2, 0), cmap= "gray" if image.shape[0] == 1 else None)
print(label)
plt.show()

In [ ]:
#| export
def prepare_dl_dist(
        cfg,
        client_id,
        fds,
        train= True,
        collator=torch.utils.data.default_collate,
):
    block = get_block(cfg= cfg, client_id= client_id, fds= fds, train= train)

    dist_sampler = torch.utils.data.distributed.DistributedSampler(block)

    data_loader = torch.utils.data.DataLoader(
        block,
        collate_fn=collator,
        sampler=dist_sampler,
        batch_size=cfg.data.get("batch_size", 32),
        drop_last=cfg.data.get("drop_last", False),
        pin_memory=cfg.data.get("pin_memory", False),
        num_workers=cfg.data.get("num_workers", 0),
        persistent_workers=(cfg.data.get("num_workers", 0) > 0) and cfg.data.get("persistent_workers", False),
    )


    return data_loader, dist_sampler

In [ ]:
#| export
def prepare_dl(cfg, client_id, fds, train=True, distributed=False):
    if distributed and torch.distributed.is_initialized():
        return prepare_dl_dist(cfg, client_id, fds, train)
    
    block = get_block(cfg, client_id, fds, train)
    dl = torch.utils.data.DataLoader(block, batch_size= cfg.data.batch_size, shuffle= True if train else False) # TODO: make sure you put batch_size to dataConfig keys
    return dl

In [ ]:
#| hide
dl = prepare_dl(cfg, client_id=15, fds= fds, train=True)
batch = next(iter(dl))
print(batch["image"].shape, batch["label"].shape)

torch.Size([128, 3, 32, 32]) torch.Size([128])


In [ ]:
#| hide
dist.init_process_group(backend="gloo", init_method="tcp://localhost:12345", rank=0, world_size=1)
dist_dl, dist_sampler = prepare_dl(cfg, client_id=15, fds= fds, train=True, distributed=True)
dist.destroy_process_group()


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


In [ ]:
#| hide
dist_sampler.set_epoch(0)
batch= next(iter(dist_dl))
print(batch["image"].shape, batch["label"].shape)

torch.Size([128, 3, 32, 32]) torch.Size([128])


In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()